In [2]:
import importlib
import sys

# ──────────── Paths (update for your machine) ────────────
ijepa_checkpoint = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/SEG-RDM/rdm/pretrained_enc_ckpts/ijepa/IN1K-vit.h.14-300e.pth.tar"
repo_root = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/StyleGAN2/seg-aware-stylegan2"
data_path = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/dataset/256/imagenet.zip"
out_base = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/StyleGAN2/seg-aware-stylegan2/outputs/debug_sam"

# SAM on-the-fly extraction (fallback for missing pre-computed)
sam_checkpoint = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/segProto/checkpoints/sam_vit_b_01ec64.pth"
# sam_cache_dir = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/sam_cache_unified"  # same dir as pre-computed!
sam_cache_dir = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/sam_temp"

# Pre-computed embeddings
# sam_npz_dir = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/sam_cache_unified"
sam_npz_dir ="/scratch/gilbreth/abelde/Thesis/StructureAwareGen/sam_temp"
ijepa_npz_dir = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/ijepa_embeddings"
origin_map_json = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/dataset/256/origin_map_imagenetdebug.json"

# RDM mixed training (set to a checkpoint path to enable, or None to skip)
rdm_checkpoint = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/SEG-RDM/rdm/rdm_out_final/4_nodes/batch_64/ijepa_and_seg_aware/checkpoint-last.pth"

# ──────────── Hyperparams ────────────
SAM_PROB = 0.5            # Network sees SAM tokens 50% of the time
SEM_MIX = 0.9
FUSION_ALPHA = 0.2
LAMBDA_SEG_ALIGN = 0.1
LAMBDA_SEG_DIVERSITY = 0.05

outdir = f"{out_base}/debug-run"

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import train
importlib.reload(train)

# ──────────── Build CLI args ────────────
sys.argv = [
    "train.py",
    "--outdir", outdir,
    "--data", data_path,
    "--subset", "20",       # LIMIT TO 20 IMAGES FOR DEBUGGING
    "--gpus", "1",
    "--cond", "1",
    "--batch", "4",
    "--kimg", "1",

    # I-JEPA options
    "--ijepa_checkpoint", ijepa_checkpoint,
    "--ijepa_lambda", "1.0",
    "--ijepa_image", "256",
    "--ijepa_input_channel", "3",
    "--ijepa_dim", "1280",
    "--ijepa_warmup_kimg", "0.5",
    "--sem_mixing_prob", str(SEM_MIX),
    "--fusion_alpha", str(FUSION_ALPHA),
    "--origin-map-json", origin_map_json,

    # Pre-computed embeddings (AlignedSegDataset loads from npz)
    "--use-seg-embeddings",
    "--sam-npz-dir", sam_npz_dir,
    "--ijepa-npz-dir", ijepa_npz_dir,

    # SAM on-the-fly fallback (extracts missing, saves to sam_cache_dir)
    "--sam-enabled", "true",
    "--sam-prob", str(SAM_PROB),
    "--sam-checkpoint", sam_checkpoint,
    "--sam-cache-dir", sam_cache_dir,
    "--sam-model-type", "vit_b",
    "--sam-max-masks", "250",
    "--sam-emb-logging", "true",

    # Alignment & diversity loss weights
    "--lambda-seg-align", str(LAMBDA_SEG_ALIGN),
    "--lambda-seg-diversity", str(LAMBDA_SEG_DIVERSITY),

    "--resume", "noresume",
]

# ── Optional: RDM mixed training ──
if rdm_checkpoint is not None:
    sys.argv += [
        "--rdm-checkpoint", rdm_checkpoint,
        "--rdm-mix-prob", "0.3",
        "--rdm-warmup-kimg", "0",
    ]

# Run

train.main(standalone_mode=False)

Using AlignedSegDataset with pre-computed embeddings
  SAM embeddings: /scratch/gilbreth/abelde/Thesis/StructureAwareGen/sam_temp
  I-JEPA embeddings: /scratch/gilbreth/abelde/Thesis/StructureAwareGen/ijepa_embeddings
  Origin map: /scratch/gilbreth/abelde/Thesis/StructureAwareGen/dataset/256/origin_map_imagenetdebug.json
  Max segments: 250
  Loaded origin_map with 5000 entries from /scratch/gilbreth/abelde/Thesis/StructureAwareGen/dataset/256/origin_map_imagenetdebug.json
  origin_map sanity check: '00000/img00000000.png' -> '0/105557' OK
AlignedSegDataset initialized:
  Images: /scratch/gilbreth/abelde/Thesis/StructureAwareGen/dataset/256/imagenet.zip
  SAM embeddings: /scratch/gilbreth/abelde/Thesis/StructureAwareGen/sam_temp
  I-JEPA embeddings: /scratch/gilbreth/abelde/Thesis/StructureAwareGen/ijepa_embeddings
  Origin map entries: 5000
  Max segments: 250
  Use labels (conditional): True
[3, 256, 256]

Training options:
{
  "num_gpus": 1,
  "image_snapshot_ticks": 50,
  "network

: 

In [ ]:
import zipfile
z=zipfile.ZipFile('/scratch/gilbreth/abelde/Thesis/StructureAwareGen/dataset/256/imagenet_debug_subset.zip')
print(z.namelist()[:20])

In [ ]:
cd /scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/StyleGAN2/seg-aware-stylegan2
python generate_origin_map.py \
    --source /scratch/gilbreth/abelde/Thesis/StructureAwareGen/dataset/imagenet_debug_subset/train \
    --dest /scratch/gilbreth/abelde/Thesis/StructureAwareGen/dataset/256/origin_map_imagenetdebug.json

In [1]:
import os
import numpy as np

npz_path = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/ijepa_embeddings/1/2470.npz"

print("File:", npz_path)
print("On-disk size (MB):", os.path.getsize(npz_path) / (1024**2))

data = np.load(npz_path, allow_pickle=True)

print("\nKeys:", data.files)

for k in data.files:
    v = data[k]
    if isinstance(v, np.ndarray):
        print(f"\n{k}:")
        print("  shape:", v.shape)
        print("  dtype:", v.dtype)
        print("  nbytes in RAM (MB):", v.nbytes / (1024**2))
        # quick stats (skip for object arrays)
        if v.dtype != object and v.size > 0:
            try:
                print("  min/max:", float(np.min(v)), float(np.max(v)))
                print("  mean/std:", float(np.mean(v)), float(np.std(v)))
            except Exception:
                pass
    else:
        print(f"\n{k}: (non-ndarray) type={type(v)}")

data.close()


File: /scratch/gilbreth/abelde/Thesis/StructureAwareGen/ijepa_embeddings/1/2470.npz
On-disk size (MB): 0.0047588348388671875

Keys: ['emb']

emb:
  shape: (1280,)
  dtype: float32
  nbytes in RAM (MB): 0.0048828125
  min/max: -3.570686101913452 2.9386324882507324
  mean/std: -7.450580596923828e-09 0.9999998807907104


In [2]:
import os
import torch

pth_path = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/SEG-RDM/rdm/rdm_out_final/4_nodes/batch_64/ijepa_and_seg_aware/checkpoint-last.pth"

print("File:", pth_path)
print("On-disk size (MB):", os.path.getsize(pth_path) / (1024**2))

ckpt = torch.load(pth_path, map_location="cpu", weights_only=False)

def summarize_obj(obj, indent=0, max_items=50):
    pad = " " * indent
    if isinstance(obj, dict):
        keys = list(obj.keys())
        print(f"{pad}dict with {len(keys)} keys")
        for k in keys[:max_items]:
            v = obj[k]
            print(f"{pad}- {k!r}: {type(v).__name__}", end="")
            # common cases
            if torch.is_tensor(v):
                print(f" | shape={tuple(v.shape)} dtype={v.dtype} device={v.device}")
            elif isinstance(v, (list, tuple)):
                print(f" | len={len(v)}")
            elif isinstance(v, dict):
                print(" | dict")
            else:
                # print small scalars/strings
                if isinstance(v, (int, float, str, bool)):
                    print(f" | value={v!r}")
                else:
                    print()
        if len(keys) > max_items:
            print(f"{pad}... (showing first {max_items} keys)")
    elif torch.is_tensor(obj):
        print(f"{pad}Tensor shape={tuple(obj.shape)} dtype={obj.dtype} device={obj.device}")
    elif isinstance(obj, (list, tuple)):
        print(f"{pad}{type(obj).__name__} len={len(obj)}")
        for i, v in enumerate(obj[:min(len(obj), 10)]):
            print(f"{pad}- [{i}] {type(v).__name__}")
    else:
        print(f"{pad}{type(obj).__name__}: {obj!r}")

print("\nTop-level object:")
summarize_obj(ckpt, indent=2)


File: /scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/SEG-RDM/rdm/rdm_out_final/4_nodes/batch_64/ijepa_and_seg_aware/checkpoint-last.pth
On-disk size (MB): 3711.553391456604

Top-level object:
  dict with 7 keys
  - 'model': OrderedDict | dict
  - 'model_ema': OrderedDict | dict
  - 'optimizer': dict | dict
  - 'epoch': int | value=28
  - 'scaler': dict | dict
  - 'args': Namespace
  - 'config': dict | dict


In [2]:
import torch

pth_path = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/SEG-RDM/rdm/rdm_out_final/4_nodes/batch_64/ijepa_and_seg_aware/checkpoint-last.pth"
ckpt = torch.load(pth_path, map_location="cpu", weights_only = False)
print("hello")
# Find the state_dict (common patterns)
state_dict = None
if isinstance(ckpt, dict):
    if "state_dict" in ckpt and isinstance(ckpt["state_dict"], dict):
        state_dict = ckpt["state_dict"]
    elif "model" in ckpt and isinstance(ckpt["model"], dict):
        state_dict = ckpt["model"]
    elif "G" in ckpt and isinstance(ckpt["G"], dict):
        state_dict = ckpt["G"]
    elif all(isinstance(v, torch.Tensor) for v in ckpt.values()):
        # checkpoint itself is a state_dict
        state_dict = ckpt

if state_dict is None:
    print("No obvious state_dict found. Top-level keys:", list(ckpt.keys()) if isinstance(ckpt, dict) else type(ckpt))
else:
    total_params = 0
    total_bytes = 0

    print("state_dict keys:", len(state_dict))
    for k, v in list(state_dict.items())[:200]:  # adjust limit
        if torch.is_tensor(v):
            n = v.numel()
            total_params += n
            total_bytes += v.element_size() * n
            print(f"{k:80s}  shape={tuple(v.shape)}  dtype={v.dtype}")
        else:
            print(f"{k:80s}  (non-tensor) type={type(v).__name__}")

    print("\nTotals:")
    print("  params:", total_params)
    print("  approx size in RAM (MB):", total_bytes / (1024**2))


hello
state_dict keys: 638
betas                                                                             shape=(1000,)  dtype=torch.float32
alphas_cumprod                                                                    shape=(1000,)  dtype=torch.float32
alphas_cumprod_prev                                                               shape=(1000,)  dtype=torch.float32
sqrt_alphas_cumprod                                                               shape=(1000,)  dtype=torch.float32
sqrt_one_minus_alphas_cumprod                                                     shape=(1000,)  dtype=torch.float32
log_one_minus_alphas_cumprod                                                      shape=(1000,)  dtype=torch.float32
sqrt_recip_alphas_cumprod                                                         shape=(1000,)  dtype=torch.float32
sqrt_recipm1_alphas_cumprod                                                       shape=(1000,)  dtype=torch.float32
posterior_variance                   

In [ ]:
!tensorboard --logdir /scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/SEG-RDM/rdm/rdm_out_final/2_nodes/a100/batch_64/ijepa_and_seg_aware --port 0


/home/abelde/.local/lib/python3.9/site-packages/tensorboard/default.py:30: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-02-15 18:32:02.104564: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771198322.257427 2602859 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771198322.300454 2602859 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771198322.680965 2602859 computation_placer.cc:177] computation placer already regis

In [4]:
import torch
from omegaconf import OmegaConf

rdm_checkpoint = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/SEG-RDM/rdm/rdm_out/4_nodes/batch_128/ijepa_h14/checkpoint-last.pth"

ckpt = torch.load(rdm_checkpoint, map_location='cpu', weights_only=False)

print("=" * 80)
print("CHECKPOINT KEYS:")
print("=" * 80)
print(ckpt.keys())
print()

print("=" * 80)
print("CONFIG STRUCTURE:")
print("=" * 80)
config = ckpt.get('config')
if config is not None:
    # Print as YAML for readability
    if isinstance(config, dict):
        print(OmegaConf.to_yaml(OmegaConf.create(config)))
    else:
        print(OmegaConf.to_yaml(config))
else:
    print("No 'config' key found!")
print()

print("=" * 80)
print("CONFIG['MODEL'] KEYS:")
print("=" * 80)
if config and 'model' in config:
    print(config['model'].keys() if hasattr(config['model'], 'keys') else dir(config['model']))
    print()
    print("CONFIG['MODEL']['PARAMS'] KEYS:")
    print("-" * 40)
    if 'params' in config['model']:
        params = config['model']['params']
        print(params.keys() if hasattr(params, 'keys') else dir(params))
        print()
        if 'pretrained_enc_config' in params:
            print("CONFIG['MODEL']['PARAMS']['PRETRAINED_ENC_CONFIG']:")
            print("-" * 40)
            print(params['pretrained_enc_config'])
else:
    print("No 'model' key in config!")

CHECKPOINT KEYS:
dict_keys(['model', 'model_ema', 'optimizer', 'epoch', 'scaler', 'args'])

CONFIG STRUCTURE:
No 'config' key found!

CONFIG['MODEL'] KEYS:
No 'model' key in config!


In [1]:
import numpy as np
data = np.load('/scratch/gilbreth/abelde/Thesis/StructureAwareGen/sam_cache_unified/1/masks_npz/5129.npz')
emb = data['emb']
print(f'Shape: {emb.shape}')
print(f'Mean: {emb.mean():.3f}, Std: {emb.std():.3f}')
# Check if segments are too similar
from sklearn.metrics.pairwise import cosine_similarity
sim = cosine_similarity(emb)
print(f'Avg pairwise similarity: {sim.mean():.3f}')

Shape: (21, 256)
Mean: 0.012, Std: 0.134
Avg pairwise similarity: 0.721


In [3]:
import numpy as np
data = np.load('/scratch/gilbreth/abelde/Thesis/StructureAwareGen/sam_cache_unified/1/masks_npz/24443.npz')
emb = data['emb']
print(f'Shape: {emb.shape}')
print(f'Mean: {emb.mean():.3f}, Std: {emb.std():.3f}')
# Check if segments are too similar
from sklearn.metrics.pairwise import cosine_similarity
sim = cosine_similarity(emb)
print(f'Avg pairwise similarity: {sim.mean():.3f}')

Shape: (100, 256)
Mean: 0.014, Std: 0.132
Avg pairwise similarity: 0.701


In [10]:
import numpy as np
import glob
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path

# Analyze first 5 classes (subfolders 0, 1, 2, 3, 4)
for class_id in range(5):
    print(f"\n{'='*60}")
    print(f"CLASS {class_id}")
    print(f"{'='*60}")
    
    similarities = []
    zero_variance_count = 0
    total_files = 0
    
    pattern = f'/scratch/gilbreth/abelde/Thesis/StructureAwareGen/sam_cache_unified/{class_id}/masks_npz/*.npz'
    npz_files = glob.glob(pattern)
    
    if len(npz_files) == 0:
        print(f"No files found for class {class_id}")
        continue
    
    for npz_file in npz_files:
        total_files += 1
        data = np.load(npz_file)
        emb = data['emb']
        
        if emb.shape[0] > 1:
            # Check for zero variance embeddings
            std_devs = emb.std(axis=1)
            if np.any(std_devs == 0):
                zero_variance_count += 1
                continue  # Skip files with constant embeddings
            
            # Use cosine similarity instead of correlation
            sim_matrix = cosine_similarity(emb)
            # Get upper triangle (excluding diagonal)
            triu_indices = np.triu_indices_from(sim_matrix, k=1)
            similarities.append(sim_matrix[triu_indices].mean())
    
    if len(similarities) == 0:
        print(f"No valid embeddings found for class {class_id}")
        continue
    
    print(f"Total files: {total_files}")
    print(f"Zero-variance files: {zero_variance_count} ({100*zero_variance_count/total_files:.1f}%)")
    print(f"Valid files: {len(similarities)}")
    print(f"\nAverage cosine similarity: {np.mean(similarities):.3f}")
    print(f"Std dev: {np.std(similarities):.3f}")
    print(f"Min: {np.min(similarities):.3f}, Max: {np.max(similarities):.3f}")
    
    # Distribution analysis
    sims = np.array(similarities)
    print(f"\nSimilarity distribution:")
    print(f"  < 0.4 (good diversity): {100*np.sum(sims < 0.4)/len(sims):.1f}%")
    print(f"  0.4-0.6 (moderate):     {100*np.sum((sims >= 0.4) & (sims < 0.6))/len(sims):.1f}%")
    print(f"  0.6-0.8 (high sim):     {100*np.sum((sims >= 0.6) & (sims < 0.8))/len(sims):.1f}%")
    print(f"  > 0.8 (collapsed):      {100*np.sum(sims >= 0.8)/len(sims):.1f}%")
    
    # Show a few example files with their similarities
    print(f"\nSample files:")
    sample_indices = np.random.choice(len(npz_files), min(3, len(npz_files)), replace=False)
    for idx in sample_indices:
        file_path = npz_files[idx]
        file_name = Path(file_path).name
        if idx < len(similarities):
            print(f"  {file_name}: similarity = {similarities[idx]:.3f}")

print(f"\n{'='*60}")
print("SUMMARY ACROSS FIRST 5 CLASSES")
print(f"{'='*60}")


CLASS 0
Total files: 828
Zero-variance files: 27 (3.3%)
Valid files: 801

Average cosine similarity: 0.701
Std dev: 0.044
Min: 0.577, Max: 0.926

Similarity distribution:
  < 0.4 (good diversity): 0.0%
  0.4-0.6 (moderate):     0.1%
  0.6-0.8 (high sim):     96.8%
  > 0.8 (collapsed):      3.1%

Sample files:
  577045.npz: similarity = 0.664
  1277622.npz: similarity = 0.650
  263616.npz: similarity = 0.652

CLASS 1
Total files: 833
Zero-variance files: 2 (0.2%)
Valid files: 830

Average cosine similarity: 0.708
Std dev: 0.050
Min: 0.589, Max: 0.980

Similarity distribution:
  < 0.4 (good diversity): 0.0%
  0.4-0.6 (moderate):     0.2%
  0.6-0.8 (high sim):     94.1%
  > 0.8 (collapsed):      5.7%

Sample files:
  1069060.npz: similarity = 0.686
  582644.npz: similarity = 0.671
  525265.npz: similarity = 0.691

CLASS 2
Total files: 520
Zero-variance files: 0 (0.0%)
Valid files: 520

Average cosine similarity: 0.726
Std dev: 0.056
Min: 0.633, Max: 0.926

Similarity distribution:
  < 0.